# Opening Distribution in the FIDE Candidates Tournament

Comparing opening choices across recent Candidates Tournaments using data from Lichess broadcasts.

In [1]:
import io
import time
import chess.pgn
import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [2]:
# Round IDs scraped from Lichess broadcast pages

OPEN_TOURNAMENTS = {
    "2022": {
        "name": "FIDE Candidates 2022 (Open)",
        "round_ids": [
            "LsFeKWZU", "sylFQGas", "oe2udItH", "0QuWnLkU", "1ZAF8srK",
            "yF4JxPcn", "OFBhwamI", "fsvj5GFW", "pk9gRSMB", "FWJYzJJJ",
            "16iH8elQ", "ZfmMC2Gh", "oIi8sTms", "ZA07lchF",
        ],
    },
    "2024": {
        "name": "FIDE Candidates 2024 (Open)",
        "round_ids": [
            "AjqSsU1w", "GenKIJ8A", "xQgaUu2y", "CPS9dENa", "MDiLWQ5M",
            "kPqEUBZJ", "X9LGGQoT", "nUycmG6L", "A7SWsX0x", "vfqUR38R",
            "46ohJ8Qt", "eJghIBZe", "Dc0DQMaQ", "S4zisI6M",
        ],
    },
    "2026": {
        "name": "FIDE Candidates 2026 (Open)",
        "round_ids": [
            "uLCZwqAK", "FRTlzP2X", "SDizieNR", "97di6JjX", "liBI9Brw",
            "WmbQrSNz", "aVbIuZ7Q", "XYRPyV8x", "hTVc2Qgj", "G3oSxPgs",
            "seMTFSPr", "9SHysZuu", "rFG1W5Tp", "oBkeHnpi",
        ],
    },
}

# Women's Candidates switched to round-robin format in 2024 (2022 was knockout)
WOMEN_TOURNAMENTS = {
    "2024": {
        "name": "FIDE Candidates 2024 (Women)",
        "round_ids": [
            "o9KMvY7j", "NLIyyGNj", "iHJboPh9", "ppaqcwoE", "IZewqxYj",
            "eNVV58I8", "lOnG4nm5", "LCuAJIx2", "qjBJs4W9", "gZXnqV5K",
            "t7qNP2lW", "nlQcOgAz", "XbBwvNRd", "7Eeze5c7",
        ],
    },
    "2026": {
        "name": "FIDE Candidates 2026 (Women)",
        "round_ids": [
            "diPdGkEA", "EMkf0c6e", "2qEm9CH3", "MDv2BlCp", "VAELuM6E",
            "hmGcNp3P", "QqBr2Kpr", "R0BP4Jy4", "6Ukl08Ir", "Es2IjSwE",
            "fDNFUpG9", "u3pemMHq", "o7DgltDn", "dllZX7eJ",
        ],
    },
}

In [3]:
def fetch_round_pgn(round_id: str) -> str:
    """Fetch PGN text for a single broadcast round from Lichess."""
    url = f"https://lichess.org/api/broadcast/round/{round_id}.pgn"
    resp = requests.get(url)
    resp.raise_for_status()
    return resp.text


def parse_games(pgn_text: str) -> list[dict]:
    """Parse all games from a PGN string and extract header info."""
    games = []
    pgn = io.StringIO(pgn_text)
    while True:
        game = chess.pgn.read_game(pgn)
        if game is None:
            break
        h = game.headers
        games.append({
            "white": h.get("White", ""),
            "black": h.get("Black", ""),
            "result": h.get("Result", ""),
            "eco": h.get("ECO", ""),
            "opening": h.get("Opening", ""),
            "date": h.get("Date", ""),
            "round": h.get("Round", ""),
        })
    return games


def fetch_all(tournaments: dict) -> pd.DataFrame:
    """Fetch and parse all games from a dict of tournaments."""
    all_games = []
    for year, info in tournaments.items():
        print(f"Fetching {info['name']}...")
        for i, rid in enumerate(info["round_ids"], 1):
            pgn_text = fetch_round_pgn(rid)
            games = parse_games(pgn_text)
            for g in games:
                g["tournament"] = year
            all_games.extend(games)
            print(f"  Round {i}: {len(games)} games")
            time.sleep(0.5)
    df = pd.DataFrame(all_games)
    df = df[df["opening"] != "?"].reset_index(drop=True)
    df["opening_family"] = df["opening"].str.split(":").str[0].str.split(",").str[0].str.strip()
    print(f"\nTotal games: {len(df)}")
    return df


def opening_chart(df, title, top_n=10):
    """Grouped bar chart: tournament on x-axis, opening family as color."""
    top_families = df["opening_family"].value_counts().head(top_n).index
    subset = df[df["opening_family"].isin(top_families)]
    ct = subset.groupby(["tournament", "opening_family"]).size().unstack(fill_value=0)
    ct = ct[top_families]
    ct_plot = ct.reset_index().melt(id_vars="tournament", var_name="opening_family", value_name="count")
    fig = px.bar(
        ct_plot, x="tournament", y="count", color="opening_family",
        barmode="group", title=title,
        labels={"count": "Number of games", "tournament": "", "opening_family": ""},
    )
    fig.update_layout(legend_title_text="")
    fig.show()
    return ct


def opening_table(ct):
    """Display the crosstab with a total column."""
    ct_display = ct.copy()
    ct_display.loc["Total"] = ct_display.sum()
    ct_display["Total"] = ct_display.sum(axis=1)
    return ct_display.T

In [4]:
df_open = fetch_all(OPEN_TOURNAMENTS)

Fetching FIDE Candidates 2022 (Open)...


  Round 1: 4 games


  Round 2: 4 games


  Round 3: 4 games


  Round 4: 4 games


...

  Round 11: 4 games


  Round 12: 4 games


  Round 13: 4 games


  Round 14: 4 games



Total games: 168


## Open Section

In [5]:
ct_open = opening_chart(df_open, "Top Opening Families by Candidates Tournament (Open)")
opening_table(ct_open)

tournament,2022,2024,2026,Total
opening_family,,,,
Ruy Lopez,13,14,0,27
Queen's Gambit Declined,1,5,20,26
Sicilian Defense,11,9,4,24
Petrov's Defense,6,7,6,19
English Opening,4,0,8,12
Italian Game,5,5,1,11
Catalan Opening,5,1,3,9
Nimzo-Indian Defense,3,2,2,7
Four Knights Game,3,1,1,5


In [6]:
df_women = fetch_all(WOMEN_TOURNAMENTS)

Fetching FIDE Candidates 2024 (Women)...


  Round 1: 4 games


  Round 2: 4 games


  Round 3: 4 games


  Round 4: 4 games


...

  Round 11: 4 games


  Round 12: 4 games


  Round 13: 4 games


  Round 14: 4 games



Total games: 112


## Women's Section

In [7]:
ct_women = opening_chart(df_women, "Top Opening Families by Candidates Tournament (Women)")
opening_table(ct_women)

tournament,2024,2026,Total
opening_family,,,
Sicilian Defense,6,13,19
Italian Game,6,11,17
Ruy Lopez,10,5,15
Queen's Gambit Declined,7,5,12
Petrov's Defense,2,2,4
Catalan Opening,4,0,4
Scotch Game,1,3,4
French Defense,2,1,3
Caro-Kann Defense,1,2,3


## Combined (Open + Women, 2024 & 2026)

In [8]:
df_combined = pd.concat([
    df_open[df_open["tournament"].isin(["2024", "2026"])],
    df_women,
], ignore_index=True)

ct_combined = opening_chart(df_combined, "Top Opening Families by Candidates Tournament (Combined)")
opening_table(ct_combined)

tournament,2024,2026,Total
opening_family,,,
Queen's Gambit Declined,12,25,37
Sicilian Defense,15,17,32
Ruy Lopez,24,5,29
Italian Game,11,12,23
Petrov's Defense,9,8,17
English Opening,0,10,10
Catalan Opening,5,3,8
French Defense,5,3,8
Nimzo-Indian Defense,3,3,6
